# Pre-process for Attrition Prediction
This notebook applies SMOTE oversampling on the processed data to balance the Attrition classes.  

In [1]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE

In [ ]:
# Load the processed dataset
processed_data = pd.read_csv('./data/processed_data.csv')
print(f"Loaded processed_data.csv — shape: {processed_data.shape}")
print(f"\nAttrition distribution (scaled values):")
print(processed_data['Attrition'].value_counts())

Loaded processed_data.csv — shape: (1470, 37)

Attrition distribution (scaled values):
Attrition
-0.439525    1232
 2.275186     238
Name: count, dtype: int64


## SMOTE Oversampling for Attrition
The Attrition class is imbalanced (~84% No vs ~16% Yes).  

In [ ]:
# Binarize Attrition for SMOTE
attrition_scaled = processed_data['Attrition'].values
attrition_binary = (attrition_scaled == attrition_scaled.max()).astype(int)

print("Before SMOTE:")
print(f"  No Attrition (0): {np.sum(attrition_binary == 0)}")
print(f"  Attrition    (1): {np.sum(attrition_binary == 1)}")
print(f"  Total:            {len(attrition_binary)}")

# Apply SMOTE
smote = SMOTE(random_state=42)
X_smote = processed_data.drop(columns=['Attrition']).values
X_resampled, y_resampled = smote.fit_resample(X_smote, attrition_binary)

print(f"\nAfter SMOTE:")
print(f"  No Attrition (0): {np.sum(y_resampled == 0)}")
print(f"  Attrition    (1): {np.sum(y_resampled == 1)}")
print(f"  Total:            {len(y_resampled)}")

Before SMOTE:
  No Attrition (0): 1232
  Attrition    (1): 238
  Total:            1470

After SMOTE:
  No Attrition (0): 1232
  Attrition    (1): 1232
  Total:            2464


In [ ]:
# Reconstruct balanced DataFrame and save
attrition_val_yes = processed_data['Attrition'].max()  
attrition_val_no = processed_data['Attrition'].min() 
attrition_resampled = np.where(y_resampled == 1, attrition_val_yes, attrition_val_no)

# Build the balanced DataFrame with all original columns
feature_cols = [c for c in processed_data.columns if c != 'Attrition']
balanced_df = pd.DataFrame(X_resampled, columns=feature_cols)
balanced_df['Attrition'] = attrition_resampled

# Reorder columns to match original
balanced_df = balanced_df[processed_data.columns]

# Save to a processed_data_attrition.csv
balanced_df.to_csv('./data/processed_data_attrition.csv', index=False)

print(f"✓ Balanced data saved to processed_data_attrition.csv — shape: {balanced_df.shape}")
print(f"  Attrition distribution:")
print(balanced_df['Attrition'].value_counts())

✓ Balanced data saved to processed_data_attrition.csv — shape: (2464, 37)
  Attrition distribution:
Attrition
 2.275186    1232
-0.439525    1232
Name: count, dtype: int64
